# Insider Signals — Backtest Analysis (2022–2024)

Evaluates the Form 4 insider-buy signal against four research questions:

- **Q1** Does the rule-filtered signal produce positive alpha over SPY?
- **Q2** Do cluster buys (≥2 insiders) outperform single-insider buys?
- **Q4** Which holding period (1w / 4w / 12w) is optimal?
- **Bonus** Does role tier (CEO/CFO vs director) predict forward returns?

Data source: `backtest_results.csv` produced by `backtest_agent.py`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('backtest_results.csv', parse_dates=['scan_date'])
df = df.dropna(subset=['ret_1w', 'ret_4w', 'ret_12w'], how='all')
print(f'Dataset: {len(df):,} signals | {df["ticker"].nunique():,} unique tickers')
print(f'Period:  {df["scan_date"].min().date()} to {df["scan_date"].max().date()}')

## Dataset Overview

In [ ]:
daily = df.groupby('scan_date').size()
print(f'Signals per day:  mean={daily.mean():.1f}  median={daily.median():.0f}  max={daily.max()}')
print(f'Cluster buys:     {df["is_cluster"].sum():,} of {len(df):,} ({df["is_cluster"].mean()*100:.1f}%)')
print()
print('Role tier breakdown:')
tier_labels = {'tier1': 'CEO/CFO/Chairman/President', 'tier2': 'EVP/SVP/VP/Chief', 'tier3': 'Director/Other'}
print(df['role_tier'].map(tier_labels).value_counts().to_string())
print()
print('Top 10 tickers by signal count:')
print(df['ticker'].value_counts().head(10).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
daily.plot(ax=ax, color='steelblue', alpha=0.25, linewidth=0.8)
daily.rolling(20).mean().plot(ax=ax, color='steelblue', linewidth=2, label='20-day rolling avg')
ax.set_title('Qualifying Form 4 signals per trading day', fontsize=13)
ax.set_ylabel('Signal count')
ax.legend()
plt.tight_layout()
plt.show()

## Q1 — Does the signal have edge over SPY?

Alpha = signal return minus SPY return over the same holding period.  
t-test null hypothesis: mean alpha = 0.

In [ ]:
rows = []
for h in ['1w', '4w', '12w']:
    alpha = df[f'alpha_{h}'].dropna()
    ret   = df[f'ret_{h}'].dropna()
    spy   = df[f'spy_ret_{h}'].dropna()
    t, p  = stats.ttest_1samp(alpha, 0)
    rows.append({
        'Horizon':        h,
        'N':              len(alpha),
        'Mean signal %':  round(ret.mean(), 2),
        'Mean SPY %':     round(spy.mean(), 2),
        'Mean alpha %':   round(alpha.mean(), 2),
        'Win rate %':     round((alpha > 0).mean() * 100, 1),
        't-stat':         round(t, 2),
        'p-value':        round(p, 4),
    })
pd.DataFrame(rows).set_index('Horizon')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h in zip(axes, ['1w', '4w', '12w']):
    data = df[f'alpha_{h}'].dropna().clip(-30, 30)
    ax.hist(data, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0,           color='red',   linewidth=1,   linestyle='--', label='Zero')
    ax.axvline(data.mean(), color='green', linewidth=1.5, linestyle='-',
               label=f'Mean={data.mean():.2f}%')
    ax.set_title(f'{h} alpha vs SPY', fontsize=12)
    ax.set_xlabel('Alpha (%)')
    ax.legend(fontsize=9)
plt.suptitle('Alpha distribution (clipped at ±30%)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## Q2 — Cluster vs single-insider buys

Mann-Whitney U one-sided test: does cluster alpha stochastically dominate single alpha?

In [ ]:
rows = []
for h in ['1w', '4w', '12w']:
    c = df[df['is_cluster']][f'alpha_{h}'].dropna()
    s = df[~df['is_cluster']][f'alpha_{h}'].dropna()
    _, p = stats.mannwhitneyu(c, s, alternative='greater')
    rows.append({
        'Horizon':                    h,
        'Cluster N':                  len(c),
        'Cluster alpha %':            round(c.mean(), 2),
        'Single N':                   len(s),
        'Single alpha %':             round(s.mean(), 2),
        'Difference (C-S) %':         round(c.mean() - s.mean(), 2),
        'p-value (cluster > single)': round(p, 4),
    })
pd.DataFrame(rows).set_index('Horizon')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, h in zip(axes, ['1w', '4w', '12w']):
    c_data = df[df['is_cluster']][f'alpha_{h}'].dropna().clip(-40, 40)
    s_data = df[~df['is_cluster']][f'alpha_{h}'].dropna().clip(-40, 40)
    bplot = ax.boxplot(
        [s_data, c_data],
        labels=['Single', 'Cluster'],
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color='crimson', linewidth=2),
    )
    colours = ['#90caf9', '#1565c0']
    for patch, colour in zip(bplot['boxes'], colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(f'{h} alpha: Cluster vs Single', fontsize=12)
    ax.set_ylabel('Alpha vs SPY (%)')
plt.tight_layout()
plt.show()

## Q4 — Which holding period is optimal?

In [ ]:
horizons = ['1w', '4w', '12w']
means  = [df[f'alpha_{h}'].mean() for h in horizons]
stds   = [df[f'alpha_{h}'].std()  for h in horizons]
ratios = [m / s if s else 0 for m, s in zip(means, stds)]
wins   = [(df[f'alpha_{h}'] > 0).mean() * 100 for h in horizons]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colour = '#1565c0'

specs = [
    (axes[0], means,  'Mean alpha by holding period',    'Alpha %',    0),
    (axes[1], ratios, 'Alpha / StdDev (signal quality)', 'Ratio',      0),
    (axes[2], wins,   'Win rate by holding period',      'Win rate %', 50),
]
for ax, vals, title, ylabel, ref in specs:
    bars = ax.bar(horizons, vals, color=colour, alpha=0.8)
    ax.axhline(ref, color='red', linewidth=1, linestyle='--')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + abs(max(vals)) * 0.02,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Bonus — Role tier breakdown

Does CEO/CFO/Chairman conviction (tier 1) outperform director/other (tier 3)?

In [ ]:
tier_labels = {
    'tier1': 'CEO/CFO/Chairman',
    'tier2': 'EVP/SVP/VP',
    'tier3': 'Director/Other',
}
rows = []
for tier, label in tier_labels.items():
    sub = df[df['role_tier'] == tier]
    for h in ['1w', '4w', '12w']:
        alpha = sub[f'alpha_{h}'].dropna()
        rows.append({
            'Role':         label,
            'Horizon':      h,
            'N':            len(alpha),
            'Mean alpha %': round(alpha.mean(), 2),
            'Win rate %':   round((alpha > 0).mean() * 100, 1),
        })
role_df = pd.DataFrame(rows)
print('Mean alpha % by role and horizon:')
display(role_df.pivot(index='Role', columns='Horizon', values='Mean alpha %'))
print()
print('Win rate % by role and horizon:')
display(role_df.pivot(index='Role', columns='Horizon', values='Win rate %'))

In [ ]:
tier_colours = {'tier1': '#1565c0', 'tier2': '#42a5f5', 'tier3': '#90caf9'}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, h in zip(axes, ['1w', '4w', '12w']):
    data = [
        df[df['role_tier'] == tier][f'alpha_{h}'].dropna().clip(-40, 40)
        for tier in tier_labels
    ]
    bplot = ax.boxplot(
        data,
        labels=list(tier_labels.values()),
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color='crimson', linewidth=2),
    )
    for patch, colour in zip(bplot['boxes'], tier_colours.values()):
        patch.set_facecolor(colour)
        patch.set_alpha(0.8)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(f'{h} alpha by role tier', fontsize=12)
    ax.set_ylabel('Alpha vs SPY (%)')
    ax.tick_params(axis='x', rotation=12)
plt.tight_layout()
plt.show()